# Иерархия доверия в киберимунных системах

## Цель проекта
Разработать шаблон для организации уровня иерархии доверия в киберимунных системах, где доверие к элементам каждого уровня иерархии основывается на цепочке доказательств их целостности и аутентичности, начиная с доверенного корневого уровня.

## Основная идея
В киберимунных системах критически важно обеспечить доверие к компонентам на всех уровнях. Иерархия доверия позволяет:

*   Определить корень доверия (Trust Anchor) — начальный уровень, которому система доверяет без дополнительных доказательств
*   Построить цепочку верификации — каждый следующий уровень доказывает свою надежность через проверку предыдущим
*   Обеспечить кибериммунность — устойчивость к компрометации за счет изоляции уровней и строгой верификации

## Детали реализации
### Уровни иерархии доверия
* Уровень 0 (Root of Trust) – Аппаратный или строго верифицированный программный модуль (например, TPM, Secure Boot).
* Уровень 1 (Core Components) – Критичные системные компоненты (гипервизор, ОС, микросервисы).
* Уровень 2 (Trusted Services) – Сервисы, работающие под контролем предыдущего уровня (например, PKI, аутентификация).
* **Уровень 3 (Applications)** – Пользовательские приложения, чья доверенность подтверждается цепочкой сертификатов.

### Механизмы проверки доверия
Целостность: Хеширование для проверки содержимого приложения и цифровые подписи для аутентификации
Аутентичность: Сертификаты X.509

Для реализации проекта мы решили построить шаблон для 3-го уровня иерархии. В качестве языка реализации был выбран Python. Мы планируем использовать сравнение хэш-суммы приложения с эталоном и последовательность аутентифицированных подписей.

Выбранный уровень не требует работы с аппаратными элементами системы на стадии нашей работы, однако в процессе имплементации предполагается взаимодействие с TPM (Trusted Platform Module).

# Архитектура решения

Для аутентификации приложения мы будем использовать цепочку сертификатов и проверку подписей:

*   Корневой сертификат (CA) — хранится в защищённом хранилище (например, TPM)
*   Сертификат разработчика — подписан CA, используется для подписи приложений
*   Приложение — содержит цифровую подпись и метаданные для проверки

Удостоверяющим цетром (CA, Certificate Authority) в нашей иерархии будет корневой сертификат, он выпускает, подписывает и отзывает остальные сертификаты. Предпологается, что именно он будет размещен на TPM, например, на чипе на плате, или на интегрированном модуле процессора.

Корневой сертификат подписывает сертификат разработчика, который, в свою очередь, подписывает приложение. Таким образом, приложение содержит цифровую подпись разработчика и метаданные для проверки.

## Процесс проверки подписи
### Перед проверкой

У СА есть закрытый ключ и публичный ключ: закрытым подписывается сертификат разработчика, а открытый будет нужен для проверки на стороне пользователя.
У разработчика также есть закрытый ключ (dev.key) и публичный ключ. Сертификат (dev.crt) содержит публичный ключ разработчика и подписан СА.
Пользователю доступен публичный ключ СА и приложение. У приложения есть файл (app.bin) и подпись (app.sig).

### Процесс проверки

Чтобы установить аутентичность приложения, пользователь проверяет сертификат (dev.crt) с помощью публичного ключа СА. Если подпись действительна, то в сертификате содержится валидный публичный ключ и можно переходить к проверке приложения. С помощью ключа расшифровываем подпись (app.sig) и получаем хэш приложения. Сравниваем с хэшем для файла (app.sig): если они совпадают, то файл не был изменен.

Таким образом на первом этапе мы проверяем, что никто не изменил сертификат разработчика, а на втором - что никто не изменил содержимое приложения.

В случае компрометации сертификата нам потребуется знать, что валидный сертификат теперь нельзя использовать для проверки приложений. Для этого мы будем обращаться к CRL (Certificate Revocation List) - списку отозванных сертификатов.

# Генерация сертификатов

Напишем запрос для генерации сертификата (корневого СА):

*   openssl req - запрос
*   -x509 - генерируем сертификат по стандарту X.509
*   -newkey	rsa:4096 - создаем новый RSA-ключ длиной 4096 бит
*   -days	365 - срок действия сертификата в днях: по истечении срока сертификат утрачивает магическую силу
*   -nodes - не шифровать приватный ключ паролем (NO DES)
*   -keyout	ca.key - файл для сохранения приватного ключа
*   -out	ca.crt - файл для сохранения сертификата

In [1]:
# Создание директорий
!mkdir -p ca certs apps demoCA/newcerts
!touch demoCA/index.txt
!echo 01 > demoCA/serial
!echo "01" > demoCA/crlnumber  # Важно! Этот файл нужен для генерации CRL!

In [2]:
!openssl req -x509 -newkey rsa:4096 -days 365 -nodes \
    -keyout ca/ca.key -out ca/root_ca.crt \
    -subj "/CN=eldar"

.+..+.+.....+......................+.....+.+..+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++*.+..+....+..............+...+....+.....+...+...+...+....+...+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++*......+....+......+...+...+....................+.......+........+.......+...............+........+....+...+..+.+.......................+......+.+..+...+...+..........+.........+..+...+.+.........+..+....+...........................+....................+....+........+..........+..+....+...........+..........+...........+...+.......+.........+............................................+.......+.....+..........+............+........+...+.+...+......+.........+......+......+........+...+....+...............+...+...+.....+.............+..+.........+...................+.....+..........+...............+.........+..+...+.......+.....+.........+.......+.................+......+.......+.....+.........+................+............+...+..............+.+.........

Этот запрос генерирует ключ и запрос на подпись (CSR) разработчика

In [3]:
!openssl req -newkey rsa:2048 -nodes \
    -keyout certs/dev.key -out certs/dev.csr \
    -subj "/C=RU/ST=Saint-Petersburg/L=Saint-Petersburg/O=spbgu/OU=IT/CN=kiracoffee228@gmail.com"

.....+.+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++*...+...+.+.........+..+.......+..+..........+...+...........+....+...+...+.....+......+....+.....+.+...........+...+.+.....+...+.......+...........+...+.........+.+......+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++*.+.....+.........+.+......+.....+....+...+.....+......+......+......+...+.+...+............+.....+.........+....+...+.........+...+.....+...+..........+...+..+...+....+..+................+..+.+.................+....+......+...+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
........+........+......+.+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++*.....+.....+.+..+............+.............+...+..+...+.+........+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++*.+....+...........+.........+.+...........+....+...+..+.........+...................+..+.+........+.+...+..+......+..........+..+.+..+............+......+............+.........

Запрос на подпись CSR с помощью корневого СА. Заметим, что здесь входным файлом будет запрос на подпись, а не готовый сертификат. Параметр -CAcreateserial создаст файл с серийным номером сертификата - он пригодится, например, для добавления сертификата в CRL

In [4]:
!openssl x509 -req -days 365 -in certs/dev.csr -CA ca/root_ca.crt -CAkey ca/ca.key -CAcreateserial -out certs/developer.crt

Certificate request self-signature ok
subject=C = RU, ST = Saint-Petersburg, L = Saint-Petersburg, O = spbgu, OU = IT, CN = kiracoffee228@gmail.com


In [5]:
# Скопировать системный конфиг в текущую папку
!cp /etc/ssl/openssl.cnf ./openssl.cnf

Команда для генерации списка отозванных сертификатов

In [6]:
!openssl ca -config openssl.cnf -gencrl -keyfile ca/ca.key -cert ca/root_ca.crt -out crl.pem

Using configuration from openssl.cnf


In [7]:
!mkdir -p apps
!echo "Test app data" > apps/secure_app.bin

In [8]:
!openssl dgst -sha256 -sign certs/dev.key -out apps/secure_app.sig apps/secure_app.bin

# Шаблон

In [9]:
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import padding, rsa
from cryptography.hazmat.primitives.serialization import load_pem_private_key
from cryptography.x509 import load_pem_x509_certificate, load_pem_x509_crl
from cryptography.x509.oid import NameOID
from cryptography.exceptions import InvalidSignature
from datetime import datetime, timedelta, timezone
import os
from unittest.mock import MagicMock, patch
import unittest
from unittest.mock import MagicMock, patch, mock_open


class TrustHierarchy:
    def __init__(self, root_ca_path, crl_path=None):
        """
        root_ca_path: Путь к корневому сертификату
        crl_path: Путь к CRL
        """
        self.root_ca = self._load_cert(root_ca_path)
        self.crl = self._load_crl(crl_path) if crl_path else None
        self.trusted_certs = [self.root_ca]

    def add_intermediate_ca(self, cert_path):
        """
        Добавляет промежуточный сертификат в цепочку доверия.

        cert_path: Путь к сертификату
        """
        cert = self._load_cert(cert_path)
        self._verify_cert_chain(cert)
        self.trusted_certs.append(cert)

    def verify_application(self, app_path, signature_path, dev_cert_path):
        """
        Полная верификация приложения.

        app_path (str): Путь к файлу приложения
        signature_path (str): Путь к файлу подписи
        dev_cert_path (str): Путь к сертификату разработчика
        """

        self._check_file_exists(app_path)
        self._check_file_exists(signature_path)
        self._check_file_exists(dev_cert_path)

        dev_cert = self._load_cert(dev_cert_path)

        self._verify_cert_chain(dev_cert)

        self._verify_signature(app_path, signature_path, dev_cert)

        self._verify_integrity(app_path)

        return True

    def _verify_cert_chain(self, cert):
        """
        Проверяет цепочку сертификатов до корневого CA.

        cert: Сертификат для проверки
        """
        current_cert = cert
        verified = False

        for trusted_cert in reversed(self.trusted_certs):
            try:
                trusted_cert.public_key().verify(
                    current_cert.signature,
                    current_cert.tbs_certificate_bytes,
                    padding.PKCS1v15(),
                    current_cert.signature_hash_algorithm
                )
                current_cert = trusted_cert
                verified = True
            except InvalidSignature:
                continue

        if not verified:
            raise TrustError("Цепочка сертификатов не верифицирована")

        self._check_cert_validity(cert)

    def _check_cert_validity(self, cert):
        """
        Проверяет срок действия и статус отзыва сертификата.

        cert: Сертификат для проверки
        """
        now = datetime.now(timezone.utc)

        if not (cert.not_valid_before_utc <= now <= cert.not_valid_after_utc):
            raise TrustError(f"Сертификат недействителен. Период: {cert.not_valid_before_utc} - {cert.not_valid_after_utc}")

        if self.crl and self.crl.get_revoked_certificate_by_serial_number(cert.serial_number):
            raise TrustError(f"Сертификат отозван (серийный номер: {cert.serial_number})")

    def _verify_signature(self, app_path, signature_path, cert):
        """
        Проверяет подпись приложения.
        """
        with open(app_path, "rb") as f:
            app_data = f.read()

        with open(signature_path, "rb") as f:
            signature = f.read()

        print(f"\n[Отладка] Размер данных: {len(app_data)} байт")
        print(f"[Отладка] Размер подписи: {len(signature)} байт")
        print(f"[Отладка] Алгоритм публичного ключа: {type(cert.public_key())}")

        try:
            # PKCS1v15
            cert.public_key().verify(
                signature,
                app_data,
                padding.PKCS1v15(),
                hashes.SHA256()
            )
            print("[Отладка] Подпись верифицирована с PKCS1v15")
        except InvalidSignature:
            try:
                # PSS
                cert.public_key().verify(
                    signature,
                    app_data,
                    padding.PSS(
                        mgf=padding.MGF1(hashes.SHA256()),
                        salt_length=padding.PSS.MAX_LENGTH
                    ),
                    hashes.SHA256()
                )
                print("[Отладка] Подпись верифицирована с PSS")
            except InvalidSignature as e:
                raise TrustError(f"Ошибка проверки подписи: {str(e)}\n"
                              f"Возможные причины:\n"
                              f"1. Файл был изменен после подписания\n"
                              f"2. Использован другой алгоритм хеширования\n"
                              f"3. Неверный сертификат для проверки")

    def _verify_integrity(self, app_path):
        """
        Проверяет целостность приложения через хеш.

        app_path: Путь к приложению
        """
        with open(app_path, "rb") as f:
            app_data = f.read()

        digest = hashes.Hash(hashes.SHA256())
        digest.update(app_data)
        file_hash = digest.finalize()

        print(f"[Проверка целостности] Хеш SHA256: {file_hash.hex()}")
        return file_hash

    def _load_cert(self, path):
        """
        Загружает сертификат из PEM-файла.
        """
        try:
            with open(path, "rb") as f:
                return load_pem_x509_certificate(f.read())
        except Exception as e:
            raise TrustError(f"Ошибка загрузки сертификата: {str(e)}")

    def _load_crl(self, path):
        """
        Загружает список отозванных сертификатов.
        """
        try:
            with open(path, "rb") as f:
                return load_pem_x509_crl(f.read())
        except Exception as e:
            raise TrustError(f"Ошибка загрузки CRL: {str(e)}")

    def _check_file_exists(self, path):
        """Проверяет существование файла."""
        if not os.path.exists(path):
            raise TrustError(f"Файл не найден: {path}")


class TrustError(Exception):
    """Класс для ошибок системы доверия."""
    pass


class TestTrustHierarchy(unittest.TestCase):
    """Unit-тесты для системы управления доверием."""

    def setUp(self):
        self.mock_root_ca = MagicMock()
        self.mock_cert = MagicMock()
        self.mock_crl = MagicMock()

        self.mock_root_ca.public_key.return_value.verify.return_value = None
        self.mock_cert.public_key.return_value.verify.return_value = None
        self.mock_crl.get_revoked_certificate_by_serial_number.return_value = None

        now = datetime.now(timezone.utc)
        self.mock_cert.not_valid_before_utc = now - timedelta(days=1)
        self.mock_cert.not_valid_after_utc = now + timedelta(days=1)
        self.mock_cert.serial_number = 12345
        self.mock_cert.tbs_certificate_bytes = b'tbs_data'
        self.mock_cert.signature = b'signature'
        self.mock_cert.signature_hash_algorithm = hashes.SHA256()

        dummy_path = open('dummy_path.crt', 'w')
        dummy_path.close()
        dummy_crl = open('dummy_crl.pem', 'w')
        dummy_crl.close()

    @patch('__main__.load_pem_x509_certificate')
    @patch('builtins.open', new_callable=mock_open)
    def test_init_with_valid_root_ca(self, mock_file, mock_load_cert):
        """Тест инициализации с валидным корневым сертификатом."""
        mock_load_cert.return_value = self.mock_root_ca
        trust = TrustHierarchy("dummy_path.crt")
        self.assertEqual(len(trust.trusted_certs), 1)

    @patch('__main__.load_pem_x509_certificate')
    @patch('builtins.open', side_effect=FileNotFoundError)
    def test_init_with_invalid_root_ca(self, mock_file, mock_load_cert):
        """Тест ошибки при загрузке невалидного корневого сертификата."""
        with self.assertRaises(TrustError):
            TrustHierarchy("invalid.crt")

    @patch('__main__.load_pem_x509_crl')
    @patch('__main__.load_pem_x509_certificate')
    @patch('builtins.open', new_callable=mock_open)
    def test_init_with_crl(self, mock_file, mock_load_cert, mock_load_crl):
        """Тест инициализации с CRL."""
        mock_load_cert.return_value = self.mock_root_ca
        mock_load_crl.return_value = self.mock_crl
        trust = TrustHierarchy("dummy_path.crt", "dummy_crl.pem")
        self.assertIsNotNone(trust.crl)

    @patch('__main__.load_pem_x509_certificate')
    @patch('builtins.open', new_callable=mock_open)
    def test_verify_expired_certificate(self, mock_file, mock_load_cert):
        """Тест верификации с просроченным сертификатом."""
        mock_load_cert.return_value = self.mock_cert
        self.mock_cert.not_valid_after_utc = datetime.now(timezone.utc) - timedelta(days=1)

        trust = TrustHierarchy("dummy_path.crt")
        with self.assertRaises(TrustError):
            trust.verify_application("app.bin", "app.sig", "dev.crt")

    @patch('__main__.load_pem_x509_crl')
    @patch('__main__.load_pem_x509_certificate')
    @patch('builtins.open', new_callable=mock_open)
    def test_verify_revoked_certificate(self, mock_file, mock_load_cert, mock_load_crl):
        """Тест верификации с отозванным сертификатом."""
        mock_load_cert.return_value = self.mock_cert
        mock_load_crl.return_value = self.mock_crl
        self.mock_crl.get_revoked_certificate_by_serial_number.return_value = MagicMock()

        trust = TrustHierarchy("dummy_path.crt", "dummy_crl.pem")
        with self.assertRaises(TrustError):
            trust.verify_application("app.bin", "app.sig", "dev.crt")

    @patch('__main__.load_pem_x509_certificate')
    @patch('builtins.open', new_callable=mock_open)
    def test_verify_invalid_signature(self, mock_file, mock_load_cert):
        """Тест верификации с невалидной подписью."""
        mock_load_cert.return_value = self.mock_cert
        self.mock_cert.public_key.return_value.verify.side_effect = InvalidSignature

        trust = TrustHierarchy("dummy_path.crt")
        with self.assertRaises(TrustError):
            trust.verify_application("app.bin", "invalid.sig", "dev.crt")

    @patch('__main__.load_pem_x509_certificate')
    @patch('builtins.open', new_callable=mock_open)
    def test_verify_corrupted_app(self, mock_file, mock_load_cert):
        """Тест верификации с измененным приложением."""
        mock_load_cert.return_value = self.mock_cert
        with patch.object(TrustHierarchy, '_verify_integrity', return_value=b'corrupted_hash'):
            trust = TrustHierarchy("dummy_path.crt")
            with self.assertRaises(TrustError):
                trust.verify_application("corrupted.bin", "app.sig", "dev.crt")

    @patch('__main__.load_pem_x509_certificate')
    @patch('builtins.open', new_callable=mock_open)
    def test_verify_invalid_cert_chain(self, mock_file, mock_load_cert):
        """Тест верификации с недоверенным сертификатом."""
        mock_load_cert.return_value = self.mock_cert
        self.mock_root_ca.public_key.return_value.verify.side_effect = InvalidSignature

        trust = TrustHierarchy("dummy_path.crt")
        with self.assertRaises(TrustError):
            trust.verify_application("app.bin", "app.sig", "untrusted.crt")

if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

........
----------------------------------------------------------------------
Ran 8 tests in 0.173s

OK
